In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
import pandas as pd
from src.graph import *
from src.instance import *
from src.experiments.experiments import *
from src.experiments.objectives import *
from src.pathinstance import new_path_instance
from src.colgensolver import ColgenSolver

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ImportError: attempted relative import with no known parent package

In [ ]:
GRAPH_CSV = "./networks/anaheim/anaheim_graph.csv"

graph_df = pd.read_csv(GRAPH_CSV, index_col=0)

graph_dict = {}
for index, row in graph_df.iterrows():
    start = str(int(row["init_node"]))
    end = str(int(row["term_node"]))
    edge = (end, float(row["free_flow_time"]))
    if start in graph_dict:
        graph_dict[start].append(edge)
    else:
        graph_dict[start] = [edge]

In [ ]:
import openmatrix as omx

DEMANDS_FILE = "./networks/anaheim/demand.omx"
DEMAND_SCALE = 0.1

file: Any = omx.open_file(DEMANDS_FILE)
demand_matrix: np.ndarray = file['matrix']

demands: Demands = {}
for i in range(demand_matrix.shape[0]):
    for j in range(demand_matrix.shape[1]):
        if i == j:
            continue
        # print(f"{i+1},{j+1}: {demand_matrix[i, j]}")
        demands[(str(i+1), str(j+1))] = demand_matrix[i, j] * DEMAND_SCALE

In [ ]:
RESULTS_FOLDER = "./results/anaheim/"
EXP_NAME = "anaheim-test"


BETA = 0.14
EXP  = 4.0

LAT_CAP = 1_000_000_0000

def capped_bpr_latency(base, flow):
    lat = base * (1 + BETA * flow ** EXP)
    if lat > LAT_CAP + flow:
        return LAT_CAP + flow
    return lat

base_vars = {
    "alpha_car": 1.0,
    "beta_car": 1.0,
    "zeta_car": (1.0, 0.5),
    "alpha_bus": 1.0,
    "beta_bus": 1.0,
    "zeta_bus": (1.0, 0.5),
    "zeta_none": (-2, 1.0),
}

params = PathInstanceParams(
    car_graph  = AdjacencyGraph(graph_dict),
    bus_graph  = AdjacencyGraph(graph_dict),
    variables  = base_vars,
    n_paths    = None,
    demands    = demands,
    max_paths  = 50,
    latency_fn = capped_bpr_latency,
    demand_fn  = None,
)

experiment = EdgeRemovalBF(
    params, EXP_NAME, Verbosity.HIGH, True, True, RESULTS_FOLDER,
    new_inst=new_path_instance, new_solver=ColgenSolver,
)

result = experiment.run_control(["None"], 1)


REMOVING: None (Control)
Solving with Colgen solver...
Outer Loop Iter: 0
Current Gap: 375929458.51839375
Added 1363 new paths
Terminating Inner Loop: Min improve achieved
Inner loops: 2
Outer Loop Iter: 1
Current Gap: 0.2021644191375727
Added 90 new paths
Terminating Inner Loop: Min improve achieved
Inner loops: 4
Outer Loop Iter: 2
Current Gap: 0.0729622143385029
Added 92 new paths
Terminating Inner Loop: Min improve achieved
Inner loops: 8
Terminating Outer: Gap minimised below threshold.
Final gap: 0.001306214739554311 (6.264693554441687e-06 + 0.0012999500459998693)


In [5]:
N_EDGES = 50
flows = result.solver.inst.get_edge_flows(result.solver.last_xs)
edges_by_flow = sorted(list(flows.items()), key=lambda x: x[1], reverse=True)
edge_list = [edge for edge, _ in edges_by_flow[:50]]
edge_list

[('25', '268'),
 ('267', '24'),
 ('268', '267'),
 ('337', '29'),
 ('33', '337'),
 ('26', '273'),
 ('273', '272'),
 ('269', '25'),
 ('272', '271'),
 ('270', '269'),
 ('271', '270'),
 ('29', '337'),
 ('337', '33'),
 ('330', '31'),
 ('336', '337'),
 ('32', '333'),
 ('333', '334'),
 ('334', '335'),
 ('335', '336'),
 ('308', '29'),
 ('268', '25'),
 ('406', '38'),
 ('329', '31'),
 ('32', '332'),
 ('30', '341'),
 ('331', '330'),
 ('332', '331'),
 ('333', '32'),
 ('334', '333'),
 ('335', '334'),
 ('336', '335'),
 ('337', '336'),
 ('24', '267'),
 ('267', '268'),
 ('341', '342'),
 ('343', '329'),
 ('342', '343'),
 ('35', '389'),
 ('28', '303'),
 ('31', '330'),
 ('389', '406'),
 ('378', '36'),
 ('332', '32'),
 ('341', '30'),
 ('36', '378'),
 ('33', '361'),
 ('303', '28'),
 ('330', '331'),
 ('331', '332'),
 ('294', '295')]

In [6]:
results = experiment.run(1, edges_to_remove=edge_list)


Running 'anaheim-test' (1 / 1)...

REMOVING: None (Control)
Solving with Colgen solver...
Outer Loop Iter: 0
Current Gap: 375929458.51839375
Added 1363 new paths
Terminating Inner Loop: Min improve achieved
Inner loops: 2
Outer Loop Iter: 1
Current Gap: 0.20216441913757263
Added 90 new paths
Terminating Inner Loop: Min improve achieved
Inner loops: 4
Outer Loop Iter: 2
Current Gap: 0.07296221433850265
Added 92 new paths
Terminating Inner Loop: Min improve achieved
Inner loops: 8
Terminating Outer: Gap minimised below threshold.
Final gap: 0.0013062147395543145 (6.264693554441687e-06 + 0.0012999500459998728)

REMOVING: ('25', '268')
Solving with Colgen solver...
Outer Loop Iter: 0
Current Gap: 371909001.9754644
Added 1364 new paths
Terminating Inner Loop: Min improve achieved
Inner loops: 2
Outer Loop Iter: 1
Current Gap: 0.1916380931874168
Added 93 new paths
Terminating Inner Loop: Min improve achieved
Inner loops: 4
Outer Loop Iter: 2
Current Gap: 0.04966621709163264
Added 37 new pat

KeyboardInterrupt: 